# Regime Parallel-Coordinates Plots (L5 — no surface kinetics)

Interactive parallel-coordinates views of the **Level 5** regime-stratified sensitivity
analysis, built **entirely from the saved CSVs** in `sa_results_L5/` — no model re-runs.

This is the L5 counterpart of `regime_parallel_coords.ipynb` (which reads `sa_results/` and
is L5L6-only). Three differences follow from the L5 study design:

- **Regimes are `oxide / metal / defect`.** The third is the *parallel* bypass through
  pinholes, cracks and grain-boundary paths, not a series resistance.
- **Outputs are `flux` / `permeability` / `PRF`.** There is no `theta` (no surface step);
  `PRF` (bare-metal flux / coated flux) is the coating-effectiveness metric instead.
- **The L5 run is isothermal at three temperatures**, so every table carries a
  `temperature_K` column. Views A/B/D are drawn at one temperature at a time — mixing them
  would put three pinned values of a non-sampled parameter on the same axes. View E uses
  temperature as the axis instead.

Views:

- **A** — samples coloured by **regime** (parameter space; the regime manifold)
- **B** — one plot **per regime**, coloured by **log10(flux)** (within-regime structure)
- **C** — **δ/floor sensitivity shift** across regimes (each line = a parameter)
- **D** — **output space** coloured by regime
- **E** — **δ/floor shift across temperature**, per regime (L5-specific)

Views C and E plot **`delta_over_floor`**, not raw δ. The dummy-parameter noise floor differs
between clusters (0.076-0.102 here) even at equal n, so raw δ is not comparable across
columns; the ratio is. **A value at or below 1.0 is not resolved** — indistinguishable from a
random input the model never saw.

Interactive: drag on any axis to **brush/filter** the lines. A standalone HTML copy of each
figure is written to `sa_results_L5/figures/`. Run with the **`mace_env`** kernel.

In [1]:
import os, sys, importlib
import numpy as np
import pandas as pd

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

import calculations.sensitivity as sens
importlib.reload(sens)
from calculations.sensitivity import (
    parallel_coordinates_samples, parallel_coordinates_sensitivity, top_drivers,
    REGIME_LABELS_L5,
)

In [2]:
RESULTS_DIR = "sa_results_L5"
FIG_DIR     = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# Which temperature to draw the per-sample views at. The L5 run is isothermal at
# 773 / 1073 / 1273 K; temperature is PINNED, not sampled, so it must not become an axis.
T_SHOW = 1273

master = pd.read_csv(os.path.join(RESULTS_DIR, "master_clusters.csv"))
routeB = pd.read_csv(os.path.join(RESULTS_DIR, "routeB_givendata.csv"))

TEMPS   = sorted(master['temperature_K'].unique())
REGIMES = list(REGIME_LABELS_L5)                      # ('oxide', 'metal', 'defect')

# Slice to one temperature for the per-sample views.
master_T = master[master['temperature_K'] == T_SHOW].reset_index(drop=True)
routeB_T = routeB[routeB['temperature_K'] == T_SHOW].reset_index(drop=True)

# Columns shown on a log10 axis (span orders of magnitude). No surface-kinetics
# parameters at L5; PRF is added because it spans decades too.
LOG_COLS = {'P_upstream', 'oxide_thickness', 'metal_thickness',
            'D_ref', 'D_ox_ref', 'K_s_ref', 'K_ox_ref',
            'grain_size', 'gb_thickness', 'lattice_density',
            'trap_dislocation_N_T', 'trap_gb_N_T', 'trap_vacancy_N_T', 'trap_carbide_N_T',
            'f_pinhole', 'f_crack', 'f_gb_defect',
            'flux', 'permeability', 'PRF', 'D_eff'}
def logsub(dims):
    return [d for d in dims if d in LOG_COLS]

print("temperatures:", TEMPS, "| drawing per-sample views at", T_SHOW, "K")
print("master rows:", len(master), f"(at {T_SHOW} K: {len(master_T)})")
print("clusters:", master_T['regime'].value_counts().to_dict())

temperatures: [773.0, 1073.0, 1273.0] | drawing per-sample views at 1273 K
master rows: 15526 (at 1273 K: 5172)
clusters: {'metal': 1747, 'defect': 1732, 'oxide': 1693}


## A — samples coloured by regime

Axes = union of each regime's top-5 δ drivers of flux (+ `log10(flux)`). Each line is one
model run; colour = regime. Brush an axis (e.g. pull `log10(f_pinhole)` to its high end) to
see which regime/flux lines survive.

In [3]:
drivers = []
for r in REGIMES:
    for p in top_drivers(routeB_T, r, 'flux', k=5):
        if p not in drivers:
            drivers.append(p)
dimsA = drivers + ['flux']

figA = parallel_coordinates_samples(
    master_T, dimsA, color_by='regime', log_dims=logsub(dimsA),
    title=f'A — L5 samples by regime @ {T_SHOW} K (top δ drivers)',
    save_html=os.path.join(FIG_DIR, 'pcp_A_regime.html'))
print("A axes:", dimsA)
figA

A axes: ['H_sol_ox', 'trap_vacancy_N_T', 'P_upstream', 'K_s_ref', 'f_pinhole', 'D_ref', 'metal_thickness', 'flux']


## B — per-regime clusters coloured by log10(flux)

One PCP per regime, axes = that regime's top-6 δ drivers, colour = `log10(flux)`. Clean
within-regime view (no cross-preset mixing).

In [4]:
figsB = {}
for r in REGIMES:
    dims = top_drivers(routeB_T, r, 'flux', k=6)
    sub  = master_T[master_T['regime'] == r]
    figsB[r] = parallel_coordinates_samples(
        sub, dims, color_by='flux', log_dims=logsub(dims),
        title=f'B — L5 {r} cluster @ {T_SHOW} K, coloured by log10(flux)',
        save_html=os.path.join(FIG_DIR, f'pcp_B_{r}.html'))
    print(f"{r:8s}: n={len(sub)}  axes={dims}")

for r in REGIMES:
    figsB[r].show()

oxide   : n=1693  axes=['H_sol_ox', 'trap_vacancy_N_T', 'P_upstream', 'K_s_ref', 'f_pinhole', 'oxide_thickness']
metal   : n=1747  axes=['K_s_ref', 'D_ref', 'P_upstream', 'trap_vacancy_N_T', 'metal_thickness', 'trap_vacancy_E_b']
defect  : n=1732  axes=['D_ref', 'H_sol_ox', 'K_s_ref', 'metal_thickness', 'P_upstream', 'trap_carbide_E_b']


## C — δ/floor sensitivity shift across regimes

Each line is a **parameter**; axes are the three regimes; height = its **`delta_over_floor`**
for log10(flux). Steeply sloped lines are parameters whose importance changes between regimes.
Colour = mean importance.

Read the 1.0 level as the noise floor: a parameter sitting near or below it in every column is
not resolved anywhere, and its slope is noise rather than physics.

In [5]:
# parameter x regime matrix of delta_over_floor at T_SHOW
matC = (routeB_T[routeB_T.metric == 'flux']
        .pivot(index='parameter', columns='regime', values='delta_over_floor')
        .reset_index())
matC = matC[['parameter'] + [r for r in REGIMES if r in matC.columns]]

figC = parallel_coordinates_sensitivity(
    matC, regimes=tuple(REGIMES),
    title=f'C — L5 δ/floor shift across regimes @ {T_SHOW} K (log10 flux)',
    save_html=os.path.join(FIG_DIR, 'pcp_C_sensitivity.html'))
print("floors at this temperature:")
print(routeB_T[routeB_T.metric == 'flux'].groupby('regime')['floor'].first().round(4).to_string())
figC

floors at this temperature:
regime
defect    0.0962
metal     0.0770
oxide     0.0827


## D — output space coloured by regime

Axes = the model outputs; colour = regime. Shows how the regimes separate in *output* space —
e.g. the oxide regime reaching PRF ~30 while metal sits at ~1.0 (no barrier), and `frac_defect`
climbing toward 1 in the defect cluster.

In [6]:
dimsD = ['flux', 'PRF', 'frac_oxide', 'frac_metal', 'frac_defect', 'permeability']
figD = parallel_coordinates_samples(
    master_T, dimsD, color_by='regime', log_dims=logsub(dimsD),
    title=f'D — L5 output space by regime @ {T_SHOW} K',
    save_html=os.path.join(FIG_DIR, 'pcp_D_outputs.html'))
figD

## E — δ/floor shift across temperature (L5-specific)

One plot per regime. Each line is a **parameter**; axes are the three temperatures; height =
`delta_over_floor` for log10(flux). This is the view the L5L6 notebook has no equivalent of,
because only the L5 study is run isothermally.

A line that rises steeply from left to right is a parameter that only becomes resolvable at
high temperature — `H_sol_ox` in the oxide regime is the clearest case. Lines hugging 1.0
across all three temperatures are noise at every temperature.

In [7]:
figsE = {}
for r in REGIMES:
    sub = routeB[(routeB.metric == 'flux') & (routeB.regime == r)]
    matE = sub.pivot(index='parameter', columns='temperature_K',
                     values='delta_over_floor')
    matE.columns = [f'{int(c)}K' for c in matE.columns]
    matE = matE.reset_index()
    tcols = tuple(f'{int(t)}K' for t in TEMPS)
    figsE[r] = parallel_coordinates_sensitivity(
        matE, regimes=tcols,
        title=f'E — L5 {r} regime: δ/floor vs temperature (log10 flux)',
        save_html=os.path.join(FIG_DIR, f'pcp_E_vs_T_{r}.html'))
    best = matE.set_index('parameter')[list(tcols)].max(axis=1).nlargest(3)
    print(f"{r:8s} top-3 by best δ/floor: {best.round(2).to_dict()}")

for r in REGIMES:
    figsE[r].show()

oxide    top-3 by best δ/floor: {'H_sol_ox': 5.43, 'P_upstream': 2.27, 'oxide_thickness': 2.04}
metal    top-3 by best δ/floor: {'K_s_ref': 3.0, 'D_ref': 2.74, 'P_upstream': 1.89}
defect   top-3 by best δ/floor: {'K_s_ref': 2.46, 'D_ref': 2.33, 'trap_vacancy_N_T': 1.88}


In [8]:
import glob
print("HTML figures written to", os.path.abspath(FIG_DIR), ":")
for f in sorted(glob.glob(os.path.join(FIG_DIR, '*.html'))):
    print("  ", os.path.basename(f), f"({os.path.getsize(f)//1024} KB)")

HTML figures written to /Users/akinyemi.az/Desktop/HE_papers/yoshi/Analytical_model/MHI_permeation/Fuerst_et_al_2024/sa_results_L5/figures :
   pcp_A_regime.html (5270 KB)
   pcp_B_defect.html (4866 KB)
   pcp_B_metal.html (4868 KB)
   pcp_B_oxide.html (4863 KB)
   pcp_C_sensitivity.html (4727 KB)
   pcp_D_outputs.html (5171 KB)
   pcp_E_vs_T_defect.html (4727 KB)
   pcp_E_vs_T_metal.html (4727 KB)
   pcp_E_vs_T_oxide.html (4727 KB)
